# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`

This notebook provides an example for loading and exploring the [FAIR² Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors](https://sen.science/doi/10.71728/senscience.qs2f-h81p) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The data is defined by a Croissant schema:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed and latest
!pip install -U mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset via Croissant schema
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Review available record sets, fields, and their `@id` identifiers.

Each entity (`RecordSet`, `Field`, `Column`) in Croissant is uniquely identified by its `@id` attribute. We will enumerate all `RecordSet` and their fields using their `@id` values.

In [ ]:
from mlcroissant._dataset.metadata.types import RecordSet, Field

# List all record sets (@id and name)
if hasattr(metadata, 'record_sets'):
    record_sets = metadata.record_sets
else:
    record_sets = getattr(metadata, 'recordSet', [])
# Fallback to searching through metadata
if not record_sets:
    # Try to extract from internal dict if necessary
    if hasattr(metadata, 'to_json'):
        js = metadata.to_json()
        record_sets = js.get('recordSet', [])

print("Available RecordSet @ids and names:")
record_set_ids = []
for rs in record_sets:
    # rs is either a RecordSet object or a dict
    if isinstance(rs, dict):
        rs_id = rs.get('@id', '')
        rs_name = rs.get('name', '')
        rs_fields = rs.get('field', [])
    else:
        rs_id = getattr(rs, '@id', '') if hasattr(rs, '@id') else getattr(rs, 'id', '')
        rs_name = getattr(rs, 'name', '')
        rs_fields = getattr(rs, 'field', [])
    print(f"- @id: {rs_id}, name: {rs_name}")
    record_set_ids.append(rs_id)
    # Print each field's @id and name
    print("  Fields:")
    for field in rs_fields:
        if isinstance(field, dict):
            f_id = field.get('@id', '')
            f_name = field.get('name', '')
        else:
            f_id = getattr(field, '@id', '') if hasattr(field, '@id') else getattr(field, 'id', '')
            f_name = getattr(field, 'name', '')
        print(f"    - @id: {f_id}, name: {f_name}")

## 3. Data Extraction

Load data from one or more record sets into pandas DataFrames. Make sure to use the record set and field `@id`s gathered above.

In [ ]:
# Choose the relevant RecordSet @id(s). For this dataset, often only one primary table exists.
# Let's assume the first record set is the primary data table.
if len(record_set_ids) == 0:
    raise ValueError('No RecordSets found in metadata.')

# For this dataset, the table will likely have an @id like:
# 'https://sen.science/doi/10.71728/senscience.qs2f-h81p#recordSet-ClinicopathologicalData'
# or similar—adjust if needed based on overview output above.
main_record_set_id = record_set_ids[0]
record_sets_to_extract = [main_record_set_id]

dataframes = {}
for record_set in record_sets_to_extract:
    records = list(dataset.records(record_set=record_set))
    dataframes[record_set] = pd.DataFrame(records)

print(f"Columns in '{main_record_set_id}':\n", dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Let's apply some standard data processing steps on a numeric field and demonstrate grouping/categorization. All field/column references use their `@id` as specified by the Croissant schema.

For this dataset, fields might include age at diagnosis (`@id`: e.g. `https://sen.science/doi/10.71728/senscience.qs2f-h81p#field-AgeAtDiagnosis`) or similar. Adjust these to match the exact `@id`s from your overview above.

In [ ]:
# Example: Suppose Age at Second CRC diagnosis is at field @id:
numeric_field_id = None
possible_numeric_ids = [col for col in dataframes[main_record_set_id].columns if 'age' in col.lower() or 'Age' in col]
if possible_numeric_ids:
    # Just pick the first one for this example
    numeric_field_id = possible_numeric_ids[0]
else:
    raise ValueError('No numeric fields related to age found. Check columns of the dataframe.')

print(f"Using numeric field @id: {numeric_field_id}")

threshold = 50  # Suppose a filter to select Age > 50
filtered_df = dataframes[main_record_set_id][dataframes[main_record_set_id][numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by a categorical field, eg. 'Sex' or 'AnatomicalLocation'
group_field_id = None
for col in dataframes[main_record_set_id].columns:
    if ('sex' in col.lower() or 'anatomic' in col.lower() or 'site' in col.lower()) and col != numeric_field_id:
        group_field_id = col
        break

if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
    print(grouped_df.head())
else:
    print('No categorical grouping field found with name "Sex" or "Anatomic" in columns.')

## 5. Visualization

Let's visualize the distribution of the selected numeric field and its relationship with a categorical group if found.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(7, 4))
sns.histplot(dataframes[main_record_set_id][numeric_field_id], bins=10, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# Boxplot by group (if group_field_id is available)
if group_field_id:
    plt.figure(figsize=(7, 4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.show()
else:
    print('No grouping categorical field for boxplot.')

## 6. Conclusion

This notebook showcased step-by-step data loading, overview, extraction, processing, and basic analysis of the FAIR² colorectal cancer survivors dataset with the `mlcroissant` library—all using the dataset's Croissant schema and entity `@id` references.

- **Data were referenced and processed by Croissant `@id`, enabling traceability and reproducibility.**
- Further analysis might explore more advanced statistical or machine learning models, inter-variable relationships, or domain-specific hypotheses depending on your research or clinical interests.

Refer to the [FAIR² dataset page](https://sen.science/doi/10.71728/senscience.qs2f-h81p) and Croissant documentation for more advanced features.
